In [43]:
import pandas as pd
import src.features as fe

In [44]:
train = pd.read_csv("../data/raw/train.csv", index_col='id')
test = pd.read_csv("../data/raw/test.csv", index_col='id')
original = pd.read_csv('../data/raw/loan_dataset_20000.csv', usecols=train.columns.tolist())

In [45]:
TARGET = 'loan_paid_back'

X = train.drop(columns=[TARGET])
y = train[TARGET]

combined = pd.concat([X, test])

NUMS = ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate']
CATS = ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose', 'grade_subgrade']

# Discretization of numerical features

In [46]:
features_to_discretize = ['annual_income', 'loan_amount']

qcut_features = fe.make_quantile_binned_features(combined, features_to_discretize, n_bins=10_000)
uniform_cut_features = fe.make_uniform_binned_features(combined, features_to_discretize, n_bins=10_000)
kmeans_cut_features = fe.make_kmeans_binned_features(combined, features_to_discretize, n_bins=10_000)
log_cut = fe.make_log_binned_features(combined, features_to_discretize, n_bins=10_000)

In [47]:
combined_discrete = pd.concat([qcut_features, uniform_cut_features, kmeans_cut_features, log_cut], axis=1)

# Count encoding

In [48]:
high_cardinality_cols = fe.determine_high_cardinality_features(combined, CATS, threshold=7)
CE_cols = high_cardinality_cols.copy()
CE_cols.append('employment_status')
print(CE_cols)
print(high_cardinality_cols)

['loan_purpose', 'grade_subgrade', 'employment_status']
['loan_purpose', 'grade_subgrade']


In [49]:
CE_cols.extend(combined_discrete.columns.tolist())
print(CE_cols)

['loan_purpose', 'grade_subgrade', 'employment_status', 'annual_income_quantile_binned', 'loan_amount_quantile_binned', 'annual_income_uniform_binned', 'loan_amount_uniform_binned', 'annual_income_kmeans_binned', 'loan_amount_kmeans_binned', 'annual_income_log_binned', 'loan_amount_log_binned']


No CE for quantile bins

In [50]:
CE_cols.remove('loan_amount_quantile_binned')
CE_cols.remove('annual_income_quantile_binned')
print(CE_cols)

['loan_purpose', 'grade_subgrade', 'employment_status', 'annual_income_uniform_binned', 'loan_amount_uniform_binned', 'annual_income_kmeans_binned', 'loan_amount_kmeans_binned', 'annual_income_log_binned', 'loan_amount_log_binned']


In [51]:
combined = pd.concat([combined, combined_discrete], axis=1)

In [53]:
ce_features = fe.make_count_features(combined, CE_cols)

In [55]:
combined = pd.concat([combined, ce_features], axis=1)

# Aggregations

In [ ]:
employment_income_agg = fe.make_aggregate_features(combined, 'employment_status', 'annual_income', ['median','mean','std'])
employment_loan_agg = fe.make_aggregate_features(combined, 'employment_status', 'loan_amount', ['median','mean','std'])

# Domain related

In [57]:
combined['default_risk'] = (combined['debt_to_income_ratio'] * 0.40 +
                            (850 - combined['credit_score']) / 850 * 0.35 +
                            combined['interest_rate'] / 100 * 0.25)
combined['loan_amount_to_interest_rate_ratio'] = combined['loan_amount'] / combined['interest_rate']
combined['credit_score_coefficient'] = combined['loan_amount_to_interest_rate_ratio'] * combined['credit_score']

# Digits